# Notebook 3: ML Regression XGBoost Time-walk Correction

The purpose of this notebook is to learn the time-walk residual from the pulse and cluster properties, subtract the prediction and measure the timing resolution on test events.

The CFD baseline uses only 1/`pulse_integral`. A tree model can also use other information such as charge sharing, position in the pixel and the incidence angle. Since more factors can be considered the model can improve on the correction made by CFD baseline.

This will involve 3 main steps;
1. Loading `hits.h5`, building the features used and splitting the data into training/validation/test sets.
2. Training XGBoost with early stoppimg on the validation set.
3. Evaluating σ_t on the test set.

## Data, features and split

The target is Δt.

There are 10 features used;
1. pulse_integral, size of the pulse in arbitrary units.
2. pixel_charge, charge in the pixel that fired in units of electrons
3. cluster_charge, total charge over all pixels in units of eelctrons
4. cluster_size, number of pixels with charge
5. col_frac, position inside the pixel [0,1]
6. row_frac, position inside the pixel [0,1]
7. angle_deg, beam incidence angle
8. v_bias, bias voltage which is constant so it doesn't carry information
9. inv_pulse_integral, 1/pulse_integral
10. charge_per_pixel, cluster_charge/cluster_size

`t_raw` and `t_true` are not part of the features.

In [1]:
import numpy as np
import pandas as pd
import h5py
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

#Load all columns of hits.h5 into a DataFrame
with h5py.File('../data/hits.h5', 'r') as f:
    data = {k: np.array(f[k]) for k in f.keys()}

df = pd.DataFrame(data)

#Remove charge-only rows
n_total = len(df)
valid = np.isfinite(df['t_raw'].values)
orig_idx = np.nonzero(valid)[0]        #Row numbers in hits.h5, saved at the end
df = df[valid].reset_index(drop=True)

print(f"Dropped {n_total - len(df)} charge-only rows (no PixelHit) "
      f"-> {len(df)} usable rows")

#Target
df['delta_t'] = df['t_raw'] - df['t_true']

#Features

df['inv_pulse_integral'] = 1.0 / df['pulse_integral']


df['charge_per_pixel'] = df['cluster_charge'] / df['cluster_size']

feature_cols = [
    'pulse_integral', 'pixel_charge', 'cluster_charge', 'cluster_size',
    'col_frac', 'row_frac', 'angle_deg', 'v_bias',
    'inv_pulse_integral', 'charge_per_pixel',
]

X = df[feature_cols].values
y = df['delta_t'].values


assert np.all(np.isfinite(X)), "Non-finite values in feature matrix"
assert np.all(np.isfinite(y)), "Non-finite values in target"

#Split 70 / 15 / 15 (train / validation / test) 
# Each row is one independent event so that a random split does not leak information between the sets
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)

print(f"train: {len(X_tr)}  val: {len(X_val)}  test: {len(X_te)}")

#Scale the features. Fitting on training set only
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_val_s, X_te_s = (scaler.transform(a) for a in (X_tr, X_val, X_te))

#Save the scaler and the feature order. This is needed to apply the model later
os.makedirs('models', exist_ok=True)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

Dropped 86 charge-only rows (no PixelHit) -> 85899 usable rows
train: 60129  val: 12885  test: 12885


## Train XGBoost and measure the resolution

The model is a gradient boosted tree regressor. It adds trees one at a time and each tree corrects the error that is left by the previous ones. The loss that is minimized is the default squared error. MAE is to decide when to stop.



In [2]:
import sys
sys.path.insert(0, '../src')  
from metrics import fit_resolution

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


model = XGBRegressor(
    n_estimators           = 3000, #Maximum number of trees
    max_depth              = 5, #Depth of each tree
    learning_rate          = 0.05, #Step size of each tree
    subsample               = 0.8, #Fraction of events used per tree
    colsample_bytree        = 0.8, #Fraction of features used per tree
    min_child_weight        = 10, #Minimum weight in a leaf
    early_stopping_rounds   = 50, #Stop after 50 trees without improvement
    eval_metric              = 'mae', #Validation metric
    random_state             = 42,
    device                   = 'cpu',
)

#Train on the training set
#Validation set only controls early stopping
model.fit(
    X_tr_s, y_tr,
    eval_set = [(X_val_s, y_val)],
    verbose  = 100,
)

print(f"Best iteration: {model.best_iteration} / {model.n_estimators}")

os.makedirs('models', exist_ok=True)
model.save_model('models/xgb_model.json')

#Apply correction to the test set
correction_te = model.predict(X_te_s)
delta_t_ml_te = y_te - correction_te

mu_ml, sigma_ml = fit_resolution(delta_t_ml_te)
mu_raw, sigma_raw = fit_resolution(y_te)

print(f"Raw (test set):        σ = {sigma_raw:.1f} ps")
print(f"XGBoost correction:    σ = {sigma_ml:.1f} ps")
print(f"Improvement over raw:  {100*(1 - sigma_ml/sigma_raw):.1f}%")

val_mae = mean_absolute_error(y_val, model.predict(X_val_s))
print(f"Validation MAE: {val_mae:.2f} ps")

[0]	validation_0-mae:56.07580


[100]	validation_0-mae:31.38938


[200]	validation_0-mae:28.07304


[300]	validation_0-mae:26.67827


[400]	validation_0-mae:26.05790


...

[2200]	validation_0-mae:25.05488


[2238]	validation_0-mae:25.05459


Best iteration: 2188 / 3000


Raw (test set):        σ = 61.1 ps
XGBoost correction:    σ = 32.8 ps
Improvement over raw:  46.2%


Validation MAE: 25.05 ps


## Save the test set results
The model was trained on arrays so the original row numbers of the test events were not kept. To get them back the same split is repeated on an index array.

In [3]:
#Recover the original indices of the test set
idx_all = np.arange(len(df))
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=42)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=42)


assert np.array_equal(X[idx_te], X_te), "index recovery doesn't match the original split!"

orig_idx_te = orig_idx[idx_te]  


os.makedirs('data/derived', exist_ok=True)

with h5py.File('data/derived/ml_correction.h5', 'w') as f:
    f.create_dataset('orig_idx', data=orig_idx_te)
    f.create_dataset('delta_t', data=y_te)              
    f.create_dataset('delta_t_ml', data=delta_t_ml_te)  

print(f"Saved {len(orig_idx_te)} test-set rows to data/derived/ml_correction.h5")

Saved 12885 test-set rows to data/derived/ml_correction.h5


Results on the test set;

| | σ_t [ps] |
|---|---|
| Raw (test set) | 61.1 |
| CFD per-angle  | 55.0 |
| XGBoost | 32.8 |

The ML correction reduces σ_t by 46.2% compared to raw data. The improvement over the CFD correction is due to the additional features used.